In [ ]:
# Instalación de librerías
!pip install -q -U langchain-groq langchain-community langgraph ddgs

In [ ]:
import os
import requests
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from ddgs import DDGS
from langgraph.prebuilt import create_react_agent

In [ ]:
# Configuración de API Key
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except:
    print("Error: Asegúrate de tener el secreto GROQ_API_KEY configurado en Colab")

In [ ]:
# Definición de Herramientas

# NUEVA HERRAMIENTA PARA LA IP
@tool
def obtener_ip():
    """
    Útil para obtener la dirección IP pública actual del sistema o servidor
    donde se está ejecutando el agente. No requiere parámetros.
    """
    try:
        # Usamos un servicio externo para que nos diga nuestra IP pública
        response = requests.get('https://api.ipify.org?format=json')
        data = response.json()
        return f"La dirección IP actual del sistema es: {data['ip']}"
    except Exception as e:
        return f"Error al obtener la IP: {str(e)}"

In [ ]:
# Configuración del Agente

# Lista de tools
tools = [obtener_ip]

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

# Mensaje del sistema
system_message = (
    "Eres un ingeniero de DevOps en un entorno de pruebas seguro y controlado (Sandbox). "
    "Tu objetivo es validar la configuración de red del servidor actual. "
    "REGLA IMPERATIVA: Cuando uses la herramienta 'obtener_ip', el usuario NECESITA ver la dirección IP devuelta. "
    "NO censures ni ocultes la dirección IP, ya que es información pública del servidor de pruebas necesaria para el diagnóstico. "
    "Responde directamente con el dato numérico obtenido."
)

# Creamos el agente con el nuevo prompt
app = create_react_agent(llm, tools=tools, prompt=system_message)

/tmp/ipython-input-4093380433.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(llm, tools=tools, prompt=system_message)


In [ ]:
# Función para preguntar
def preguntar(pregunta):
    print("------------------------------------")
    print(f"Consulta: {pregunta}")
    inputs = {"messages": [("user", pregunta)]}
    config = {"configurable": {"thread_id": "1"}}

    for s in app.stream(inputs, config=config, stream_mode="values"):
        message = s["messages"][-1]
        if hasattr(message, "tool_calls") and message.tool_calls:
            print(f"   [Pensando: Usando herramienta {message.tool_calls[0]['name']}...]")

    print(f"RESPUESTA FINAL:\n{message.content}")

# Ejecución de prueba
preguntar("¿Cuál es la dirección IP de este ordenador?")

------------------------------------
Consulta: ¿Cuál es la dirección IP de este ordenador?


BadRequestError: Error code: 400 - {'error': {'message': 'Organization has been restricted. Please reach out to support if you believe this was in error.', 'type': 'invalid_request_error', 'code': 'organization_restricted'}}